<a href="https://colab.research.google.com/github/MananAslamDev/ML-Stuff/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "Content ranking in AI overview features retains baseline traffic 15% longer than standard search results."

Methodology Question: Does the validation design carry the claim regarding causality? High-authority enterprise sites naturally trigger AI overviews more often than smaller sites. If the control group wasn't stratified by domain authority, the 15% retention might just be a reflection of the brand's overall strength, rather than the AI feature itself.

Finding 2: "Pages that undergo CTR-optimization see an average 30% jump in measured sessions within 14 days."

Methodology Question: Where exactly does the "CTR-optimization" label come from? If the label is applied after a traffic spike is observed (look-ahead bias), it leaks future performance into the historical training data. The label must originate purely from a timestamped CMS edit, independent of traffic outcomes.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Authenticate with Hugging Face securely
hf_token = userdata.get('HF_TOKEN')
conn = duckdb.connect()
conn.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 2. Query the mid-panel data, pulling client_hash_id for our grouped split
query = """
    SELECT
        client_hash_id,
        gsc_impressions,
        ga4_sessions,
        ai_gemini,
        ai_copilot
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_impressions IS NOT NULL
"""
df = conn.execute(query).df().fillna(0)

# 3. Setup Target (Sessions > 0) and Features
X = df[['gsc_impressions', 'ai_gemini', 'ai_copilot']]
y = (df['ga4_sessions'] > 0).astype(int)
groups = df['client_hash_id']

# --- BEFORE: Random Split (The Leakage Trap) ---
# This allows the model to memorize specific clients rather than learning SEO patterns.
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)

rf_rand = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_rand.fit(X_train_rand, y_train_rand)
rand_preds = rf_rand.predict(X_test_rand)
rand_prec = precision_score(y_test_rand, rand_preds, zero_division=0)

# --- AFTER: Honest Grouped Split (Real-World Generalization) ---
# Entire clients are held out of the training set. The model must predict on unseen entities.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)
grp_preds = rf_grp.predict(X_test_grp)
grp_prec = precision_score(y_test_grp, grp_preds, zero_division=0)

# --- COMPARISON ---
print("--- SPLIT COMPARISON ---")
print(f"Random Split Precision (Overly Optimistic): {rand_prec:.3f}")
print(f"Grouped Split Precision (Honest Baseline):  {grp_prec:.3f}")

--- SPLIT COMPARISON ---
Random Split Precision (Overly Optimistic): 0.991
Grouped Split Precision (Honest Baseline):  0.935


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SPLIT COMPARISON ---
Random Split Precision (Overly Optimistic): 0.991
Grouped Split Precision (Honest Baseline):  0.935


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Audit Results: Final Feature Set is Clean.

Target Leakage: The target (ga4_sessions > 0) is fully isolated. We avoided including highly correlated downstream metrics (like time-on-page) in our feature set, which would have leaked the answer.

Future Leakage: All features (gsc_impressions, ai_gemini, ai_copilot) represent daily measurements recorded concurrently with the target. There is no look-ahead window exposing April data to the March training set.

Identity Leakage: client_hash_id is strictly utilized for the GroupShuffleSplit boundaries and is explicitly excluded from the X feature matrix. The model cannot memorize high-traffic domains.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Bold Claim:
"Our random forest model proves that AI visibility drives organic traffic. You must rewrite these specific zero-click pages because the model guarantees they will get traffic once optimized."

Rewritten Honest Claim:
"Our model provides decision-support by highlighting URLs where high search visibility is observed, yet zero sessions are measured. The model indicates a directional relationship between AI visibility signals and baseline traffic expectations, helping editors prioritize CTR-optimization targets."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.